# scqubits example: when does parallelizing a sweep help? (`num_cpus`, BLAS threads)

J. Koch and P. Groszkowski

For further documentation of scqubits see https://scqubits.readthedocs.io/en/latest/.

---

`ParameterSweep` can compute a sweep across worker processes via its `num_cpus`
argument. **Parallelism is not free, and very often it does *not* help — sometimes it
makes a sweep slower.** Whether it pays off is governed by one simple balance:

> parallelism helps only when&nbsp;&nbsp;**(number of grid points) × (cost per point)&nbsp;&nbsp;≫&nbsp;&nbsp;the fixed overhead**
> of starting worker processes and pickling each task to them.

So the behaviour you should *expect*:

- **Small grids, or cheap-per-point systems** (small Hilbert spaces): `num_cpus > 1`
  gives little or no speedup, and frequently a **slowdown**. This is normal — keep the
  default `num_cpus = 1`.
- **Large grids of expensive points** (big composite Hilbert spaces): this is where
  workers pay off.

A second knob interacts with the first: every eigensolve runs on a multithreaded
**BLAS/LAPACK** backend. If you run several workers and let each use all cores, you
oversubscribe — which on large dense matrices is not a small slowdown but a
**catastrophe** (an example below is ~90× slower than serial). So we cap BLAS threads
**first**, before importing anything. Timings are machine-specific — run the cells
yourself; the illustrative numbers are from a 10-core laptop and yours will differ.


## Step 0 (do this first): cap BLAS threads *before* importing scqubits

The BLAS backend reads its thread count **once, at import time**. So this must be the
**first cell you run in a fresh kernel** — before `numpy` or `scqubits` is imported. (If
scqubits is already imported in this kernel, restart it and run this first.) Capping to
`1` is a good starting point for qubit sweeps and is what prevents oversubscription once
`num_cpus > 1`. We explain *why this matters so much* in the 'BLAS footgun' section below.


In [ ]:
import os

# MUST run before numpy / scqubits are imported (restart the kernel otherwise).
for _var in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_var] = "1"


### Confirm the cap took effect

The single most common cause of confusing sweep timings is that the cap **did not
apply** — because `numpy`/`scqubits` were already imported when the env vars were set.
Check it (needs the small `threadpoolctl` package, `pip install threadpoolctl`); each
reported thread count should be `1`:


In [ ]:
import numpy as np
import scqubits as scq  # imports scipy, whose BLAS the eigensolvers use

try:
    from threadpoolctl import threadpool_info

    pools = threadpool_info()
    for pool in pools:
        print(f"{pool['internal_api']:>10}: {pool['num_threads']} thread(s)")
    if not pools:
        print("(no controllable BLAS pool detected)")
    print("\nEach count should be 1. If not, restart the kernel and run the cap cell")
    print("FIRST. (On Apple Silicon, numpy itself uses Accelerate, which ignores these")
    print(" variables; scipy's bundled OpenBLAS -- shown here -- is what scqubits uses.)")
except ImportError:
    print("threadpoolctl not installed (pip install threadpoolctl) -- skipping check.")


## A small timing helper

Wall-clock timing is noisy, so we take the median of a few repeats and discard a warm-up
run (the first parallel run pays a one-time process-startup cost).


In [ ]:
import time



def time_run(make_sweep, num_cpus, repeats=3):
    """Median wall time of one full sweep, after a discarded warm-up run.

    `make_sweep(num_cpus)` builds *and runs* a fresh ParameterSweep (autorun=True).
    """
    make_sweep(num_cpus)  # warm-up (discarded: pays one-time process startup)
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        make_sweep(num_cpus)
        times.append(time.perf_counter() - start)
    return float(np.median(times))


cores = os.cpu_count() or 1


## Demo 1 — when parallelism does *not* help (the common case)

Three coupled tunable transmons, swept over the flux of the first. With
`truncated_dim = 6` the dressed Hilbert space has dimension 6³ = 216 and each point costs
only a few milliseconds — far below the per-task overhead.


In [ ]:
def make_sweep_light(num_cpus, n_points=96, truncated_dim=6):
    qubits = [
        scq.TunableTransmon(
            EJmax=30.0, EC=0.2, d=0.1, flux=0.0, ng=0.0, ncut=50,
            truncated_dim=truncated_dim, id_str=f"tmon{i}",
        )
        for i in range(3)
    ]
    hs = scq.HilbertSpace(qubits)
    for i in range(2):
        hs.add_interaction(g_strength=0.1, op1=qubits[i].n_operator, op2=qubits[i + 1].n_operator)
    flux_vals = np.linspace(0.0, 0.5, n_points)

    def update(flux):
        qubits[0].flux = flux

    return scq.ParameterSweep(
        hilbertspace=hs, paramvals_by_name={"flux": flux_vals},
        update_hilbertspace=update, evals_count=20, num_cpus=num_cpus, autorun=True,
    )


for n in [c for c in (1, 2, 4) if c <= cores]:
    print(f"num_cpus={n}:  {time_run(make_sweep_light, n):.3f} s (median)")


**What to expect:** roughly *flat* timings — `num_cpus = 4` is no faster than `1`, and
`num_cpus = 2` may even be slower. On the reference 10-core laptop (dim 216, 96 points,
BLAS capped to 1):

| num_cpus | wall time | speedup |
|---:|---:|---:|
| 1 | 0.51 s | 1.00× |
| 4 | 0.51 s | 1.00× |

This is the **normal** outcome for a modest sweep: at ~5 ms/point the cost of pickling
each task to a worker is comparable to the work itself, so spreading the points across
processes buys nothing. **This is not a bug — it is the expected regime, and
`num_cpus = 1` (the default) is the right choice here.**


## Demo 2 — when parallelism *does* help

The *only* change: `truncated_dim = 8`, so the dressed dimension becomes 8³ = 512 and
each eigensolve now costs ~40 ms — large enough to amortize the per-task overhead.


In [ ]:
def make_sweep_heavy(num_cpus, n_points=96):
    return make_sweep_light(num_cpus, n_points=n_points, truncated_dim=8)


for n in [c for c in (1, 2, 4) if c <= cores]:
    print(f"num_cpus={n}:  {time_run(make_sweep_heavy, n):.3f} s (median)")


**What to expect:** now the workers pay off. On the reference laptop (dim 512, 96
points, BLAS capped to 1):

| num_cpus | wall time | speedup |
|---:|---:|---:|
| 1 | 3.62 s | 1.00× |
| 4 | 1.30 s | **2.8×** |

Nothing changed except the **cost per point** (~5 ms → ~40 ms, via `truncated_dim`).
That is the whole story: parallelism helps once *grid size × per-point cost* clears the
overhead. If your sweep looks like Demo 1, raising `num_cpus` will not help — make the
grid much larger, or accept that the sweep is simply fast enough serially.


## The BLAS-thread footgun — *why* we capped first

Each eigensolve runs on a multithreaded BLAS backend that by default uses *all* cores.
Run `num_cpus` workers and each launches a full BLAS pool, so you oversubscribe the cores
by a factor of `num_cpus`. On small matrices this is wasteful; on **large dense**
matrices it is *catastrophic*. Measured on the reference laptop — 5 capacitively coupled
fluxonia, dressed dim 3125, dense diagonalization, 16-point sweep:

| configuration | wall time |
|---|---:|
| `num_cpus=1` | 42 s |
| `num_cpus=4`, BLAS **uncapped** | **3608 s**  (≈ 90× *slower*) |
| `num_cpus=4`, BLAS capped to 1 | 40 s |
| `num_cpus=8`, BLAS capped to 1 | 28 s |

40 threads fighting over 10 cores on dense LAPACK collapses performance — the cap is the
difference between working and broken. (If you skip Step 0 and rerun Demo 2 with the cap
off, you will reproduce a milder version of this: `num_cpus = 2`/`4` become *much* slower
than `1`.) That is also the usual reason a `num_cpus` comparison looks 'inconclusive' or
backwards: the cap was never in effect.

**Caveat — `1` is not universally optimal.** On small matrices a single BLAS thread is
best; on large dense matrices each eigensolve benefits from several threads, so the
fastest setting balances the knobs: roughly **`num_cpus × BLAS-threads ≈ cores`**. The
optimum depends on machine, BLAS library, and matrix size — measure it.


## For large composite systems, try sparse diagonalization first

Before reaching for `num_cpus`, note that for large coupled systems the dominant cost is
the *per-point diagonalization*, and that is usually a bigger lever than parallelism.
Recent scqubits automatically uses **sparse** diagonalization (`scipy.eigsh`) for the
default method when only a few eigenstates of a large Hilbert space are requested
(`scqubits.settings.AUTO_SPARSE_DIAG`). For the 5-fluxonia system above (dressed dim
3125) this alone is ~16× faster per point than dense — far more than the ~1.5× that
multiprocessing buys there. And once sparse makes each point cheap, `num_cpus > 1` helps
even less. **Try sparse first; parallelize second.**


## Automated tuning and summary

Because the optimum depends on your machine, BLAS library, and problem, the most
reliable approach is to measure. From the scqubits **source tree**, the script
`tools/autotune_multiprocessing.py` searches the `num_cpus × BLAS-threads` frontier and
reports the fastest configuration for your system.

### Summary

- **Default `num_cpus = 1`.** Parallelism helps only when *grid size × cost-per-point*
  greatly exceeds the per-task overhead; on small or cheap sweeps it does nothing, or
  slows you down — expected, not a bug (Demo 1 vs Demo 2).
- **Cap BLAS threads before importing scqubits**, and verify it took effect.
  Oversubscription is catastrophic on large dense matrices (the 90× example). Keep
  `num_cpus × BLAS-threads ≈ cores`.
- For large composite Hilbert spaces, **sparse diagonalization is usually a bigger lever
  than multiprocessing** — try it first.
- **Measure on your own hardware** — every number here is machine-specific.
